In [ ]:
import json
import requests
from datetime import datetime, date

class AILifeOperator:
    def __init__(self):

        try:
            with open("tasks.json", "r") as file:
                self.tasks = json.load(file)
        except FileNotFoundError:
            self.tasks = []

    def save_tasks(self):
        with open("tasks.json", "w") as file:
            json.dump(self.tasks, file, indent=4)

    def add_task(self, task, priority, due):

        if task == "":
            print("Task cannot be empty.")
            return

        new_task = {
            "task": task,
            "priority": priority,
            "due": due
        }

        self.tasks.append(new_task)
        self.save_tasks()
        print("Task saved.")

    def show_task(self):
        if len(self.tasks) == 0:
            print("No tasks available.")
            return

        print("\nYour Tasks:\n")

        for i, task in enumerate(self.tasks, start=1):
            print(
                f"{i}. {task['task']}"
                f" (priority: {task['priority']})"
                f" (due: {task['due']})"
            )

    def get_due_date(self):
        today = date.today()

        due_messages = []

        for task in self.tasks:

            try:
                due_date = datetime.strptime(
                    task['due'],
                    "%Y-%m-%d"
                ).date()

                days_left = (due_date - today).days

                if days_left < 0:
                    due_messages.append(
                        f"{task['task']} is overdue!"
                    )

                elif days_left == 0:
                    due_messages.append(
                        f"{task['task']} is due today!"
                    )

                elif days_left == 1:
                    due_messages.append(
                        f"{task['task']} is due tomorrow!"
                    )

                else:
                    due_messages.append(
                        f"{task['task']} is due in {days_left} days!"
                    )

            except ValueError:
                due_messages.append(
                    f"Invalid date for {task['task']}"
                )

        return due_messages

    def ask_ai(self, user_input):
        due_info = self.get_due_date()

        prompt = f"""
  You are my personal AI Life Operator.

  curren Tasks:
  {self.tasks}

   Due Date Alerts:
   {due_info}

  Today's Date:
  {date.today()}


  The user may ask:
  - create timetable
  - organize tasks
  - suggest priorities
  - suggest prioritues
  - plan the day based on my tasks
  - motivate
  - study planning
  - productivity advice


  Always use the tasks list when needed.

  priority rules:
  - high = moat immportant
  - medium = normal important
  - low = least important

  Due date rules:
  - overdue tasks come first
  - tasks due today come next
  - task due tomorrow come next
  - then sort by priority

  When user asks:
  - what should i do first
  - organize tasks
  - make timetable
  - plan day

  Always prioritize:
  1. Overdue tasks
  2. Due today
  3. Due tomorrow
  4. High priority
  5. Medium priority
  6. Low priority

  User message:
  {user_input}
  """

        response = requests.post(
            "http://localhost:11434/api/generate",

            json={
                "model": "phi3",
                "prompt": prompt,
                "stream": False
            }
        )

        return response.json()["response"]

    def run(self):

        print("=== AI LIFE OPERATOE ===")

        print("\nCommands:")
        print("add task <task>")
        print("show task")
        print("exit")

        while True:

            user_input = input("\nYou: ")

            if user_input.lower() == "exit":

                print("Goodbye!")
                break

            elif user_input.lower().startswith("add task"):

                task = user_input[8:].strip()

                priority = input(
                    "Enter priority (high/medium/low): "
                ).lower()

                due = input(
                    "Enter due date (YYYY-MM-DD): "
                )

                self.add_task(
                    task,
                    priority,
                    due
                )

            elif user_input.lower() == "show task":
                self.show_task()

            else:
                ai_reply = self.ask_ai(user_input)

                print("\nAI:\n")

                print(ai_reply)

if __name__ == "__main__":
    assistant = AILifeOperator()
    assistant.run()

=== AI LIFE OPERATOE ===

Commands:
add task <task>
show task
exit
